# Análisis de Audio con Basic-Pitch (Spotify)
Extrae notas, pitch y genera un archivo MIDI a partir de cualquier grabación de audio.  
Ideal para analizar quenas, pincuyos y traversas.

In [1]:
# Verificar que basic-pitch está instalado correctamente
import importlib.metadata
try:
    version = importlib.metadata.version("basic-pitch")
    print(f"✓ basic-pitch versión: {version}")
except importlib.metadata.PackageNotFoundError:
    print("⚠ basic-pitch no está instalado en este kernel.")
    print("  Ejecuta: pip install basic-pitch")


✓ basic-pitch versión: 0.4.0


## 1. Importaciones

In [2]:
import os
import numpy as np
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH
import pretty_midi

## 2. Configuración del archivo de audio
Cambia `AUDIO_FILE` por la ruta a tu grabación (`.wav`, `.mp3`, `.ogg`, `.flac`).

In [3]:
# =====================================================================
# CONFIGURA AQUÍ TU ARCHIVO DE AUDIO
# =====================================================================
AUDIO_FILE = "7. Juni um 21-51.m4a"   # grabación adjunta

# Parámetros de detección (ajustables)
ONSET_THRESHOLD = 0.5    # sensibilidad para detectar inicio de nota (0-1)
FRAME_THRESHOLD = 0.3    # sensibilidad por frame (0-1)
MIN_NOTE_LEN    = 127    # duración mínima de nota en ms
MIN_FREQ        = 200.0  # frecuencia mínima en Hz (filtra ruido bajo)
MAX_FREQ        = 2000.0 # frecuencia máxima en Hz

# Verificar que el archivo existe
if not os.path.exists(AUDIO_FILE):
    print(f"⚠ Archivo '{AUDIO_FILE}' no encontrado.")
    print("  Coloca tu grabación en la carpeta del proyecto y actualiza AUDIO_FILE.")
else:
    print(f"✓ Archivo encontrado: {AUDIO_FILE}")


✓ Archivo encontrado: 7. Juni um 21-51.m4a


In [8]:
import subprocess
import imageio_ffmpeg

FFMPEG_EXE = imageio_ffmpeg.get_ffmpeg_exe()
print(f"ffmpeg: {FFMPEG_EXE}")

# Convertir m4a → wav si es necesario
SRC_FILE = "7. Juni um 21-51.m4a"
WAV_FILE  = "7. Juni um 21-51.wav"

if not os.path.exists(WAV_FILE):
    print(f"Convirtiendo {SRC_FILE} → {WAV_FILE} ...")
    result = subprocess.run(
        [FFMPEG_EXE, "-y", "-i", SRC_FILE, "-ar", "22050", "-ac", "1", WAV_FILE],
        capture_output=True, text=True, encoding="utf-8", errors="replace"
    )
    if result.returncode != 0:
        print("Error:", result.stderr[-600:])
    else:
        print(f"✓ Convertido: {WAV_FILE}")
else:
    print(f"✓ WAV ya existe: {WAV_FILE}")

AUDIO_FILE = WAV_FILE
print(f"Usando: {AUDIO_FILE}")


ffmpeg: c:\Users\CESAR FERNANDO\.conda\envs\Musik_Allgemain\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe
Convirtiendo 7. Juni um 21-51.m4a → 7. Juni um 21-51.wav ...
✓ Convertido: 7. Juni um 21-51.wav
Usando: 7. Juni um 21-51.wav


## 3. Análisis con Basic-Pitch

In [9]:
if os.path.exists(AUDIO_FILE):
    print("Analizando audio... (puede tardar unos segundos)\n")

    model_output, midi_data, note_events = predict(
        AUDIO_FILE,
        onset_threshold=ONSET_THRESHOLD,
        frame_threshold=FRAME_THRESHOLD,
        minimum_note_length=MIN_NOTE_LEN,
        minimum_frequency=MIN_FREQ,
        maximum_frequency=MAX_FREQ,
    )

    print(f"✓ Análisis completado. Notas detectadas: {len(note_events)}\n")
    print(f"{'#':<5} {'Inicio(s)':<12} {'Fin(s)':<10} {'MIDI':<8} {'Freq(Hz)':<12} {'Vel.':<6}")
    print("-" * 55)
    for i, (start, end, pitch, vel, _) in enumerate(note_events):
        freq = 440.0 * (2.0 ** ((pitch - 69) / 12.0))
        print(f"{i+1:<5} {start:<12.3f} {end:<10.3f} {pitch:<8} {freq:<12.1f} {vel:.2f}")
else:
    print("⚠ Saltando análisis: archivo de audio no disponible.")
    note_events = []
    midi_data = None

Analizando audio... (puede tardar unos segundos)

Predicting MIDI for 7. Juni um 21-51.wav...
✓ Análisis completado. Notas detectadas: 82

#     Inicio(s)    Fin(s)     MIDI     Freq(Hz)     Vel.  
-------------------------------------------------------
1     40.277       40.777     88       1318.5       0.62
2     39.941       40.277     88       1318.5       0.66
3     39.743       39.952     90       1480.0       0.33
4     39.696       39.941     88       1318.5       0.38
5     39.545       39.696     88       1318.5       0.37
6     38.442       38.639     93       1760.0       0.55
7     37.465       37.605     93       1760.0       0.45
8     36.699       36.954     91       1568.0       0.60
9     36.211       36.420     83       987.8        0.50
10    35.525       35.723     88       1318.5       0.55
11    35.096       35.270     88       1318.5       0.43
12    34.840       35.084     91       1568.0       0.57
13    33.935       34.155     83       987.8        0.59
14   

## 4. Convertir notas MIDI a nombres musicales

In [10]:
NOTAS_ES = ["Do", "Do#", "Re", "Re#", "Mi", "Fa",
            "Fa#", "Sol", "Sol#", "La", "La#", "Si"]

def midi_a_nombre(midi_num):
    nombre = NOTAS_ES[midi_num % 12]
    octava = (midi_num // 12) - 1
    return f"{nombre}{octava}"

if note_events:
    print(f"\n{'#':<5} {'Nota':<8} {'Inicio(s)':<12} {'Duración(ms)':<15} {'Freq(Hz)':<12}")
    print("-" * 55)
    for i, (start, end, pitch, vel, _) in enumerate(note_events):
        freq   = 440.0 * (2.0 ** ((pitch - 69) / 12.0))
        dur_ms = (end - start) * 1000
        print(f"{i+1:<5} {midi_a_nombre(int(pitch)):<8} {start:<12.3f} {dur_ms:<15.0f} {freq:<12.1f}")
else:
    print("Sin notas detectadas.")


#     Nota     Inicio(s)    Duración(ms)    Freq(Hz)    
-------------------------------------------------------
1     Mi6      40.277       499             1318.5      
2     Mi6      39.941       337             1318.5      
3     Fa#6     39.743       209             1480.0      
4     Mi6      39.696       245             1318.5      
5     Mi6      39.545       151             1318.5      
6     La6      38.442       197             1760.0      
7     La6      37.465       139             1760.0      
8     Sol6     36.699       255             1568.0      
9     Si5      36.211       209             987.8       
10    Mi6      35.525       197             1318.5      
11    Mi6      35.096       174             1318.5      
12    Sol6     34.840       244             1568.0      
13    Si5      33.935       221             987.8       
14    Mi6      33.608       163             1318.5      
15    Sol6     32.993       279             1568.0      
16    La6      32.796       139

## 5. Guardar el resultado como archivo MIDI

In [11]:
if midi_data is not None:
    OUTPUT_MIDI = AUDIO_FILE.rsplit(".", 1)[0] + "_notas.mid"
    midi_data.write(OUTPUT_MIDI)
    print(f"✓ MIDI guardado en: {OUTPUT_MIDI}")
    print("  Puedes abrirlo en MuseScore, GarageBand o importarlo con music21.")
else:
    print("Sin datos MIDI para guardar.")

✓ MIDI guardado en: 7. Juni um 21-51_notas.mid
  Puedes abrirlo en MuseScore, GarageBand o importarlo con music21.


## 6. Estadísticas de la grabación

In [12]:
if note_events:
    from collections import Counter

    pitches   = [int(n[2]) for n in note_events]
    freqs     = [440.0 * (2.0 ** ((p - 69) / 12.0)) for p in pitches]
    durations = [(n[1] - n[0]) * 1000 for n in note_events]
    conteo    = Counter([midi_a_nombre(p) for p in pitches])

    print("=== ESTADÍSTICAS ===")
    print(f"  Total de notas        : {len(note_events)}")
    print(f"  Nota más grave        : {midi_a_nombre(min(pitches))}  ({min(freqs):.1f} Hz)")
    print(f"  Nota más aguda        : {midi_a_nombre(max(pitches))}  ({max(freqs):.1f} Hz)")
    print(f"  Duración media (ms)   : {np.mean(durations):.0f}")
    print(f"  Duración mínima (ms)  : {np.min(durations):.0f}")
    print(f"  Duración máxima (ms)  : {np.max(durations):.0f}")
    print(f"\n  Notas más frecuentes:")
    for nota, cnt in conteo.most_common(8):
        barra = "█" * cnt
        print(f"    {nota:<6} {barra}  ({cnt})")
else:
    print("Sin datos para estadísticas.")

=== ESTADÍSTICAS ===
  Total de notas        : 82
  Nota más grave        : La5  (880.0 Hz)
  Nota más aguda        : La6  (1760.0 Hz)
  Duración media (ms)   : 213
  Duración mínima (ms)  : 139
  Duración máxima (ms)  : 546

  Notas más frecuentes:
    Mi6    ███████████████████████  (23)
    Sol6   ███████████████████  (19)
    La6    █████████████████  (17)
    Si5    ████████████  (12)
    Fa#6   ██████  (6)
    Re6    ███  (3)
    La5    ██  (2)


## 7. Exportar lista de notas a CSV

In [13]:
import csv

if note_events:
    OUTPUT_CSV = AUDIO_FILE.rsplit(".", 1)[0] + "_notas.csv"
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["#", "Nota", "MIDI", "Freq_Hz",
                         "Inicio_s", "Fin_s", "Duracion_ms", "Velocidad"])
        for i, (start, end, pitch, vel, _) in enumerate(note_events):
            freq   = 440.0 * (2.0 ** ((pitch - 69) / 12.0))
            dur_ms = (end - start) * 1000
            writer.writerow([i+1, midi_a_nombre(int(pitch)), int(pitch),
                             round(freq, 2), round(start, 3), round(end, 3),
                             round(dur_ms, 0), round(vel, 3)])
    print(f"✓ CSV guardado en: {OUTPUT_CSV}")
else:
    print("Sin notas para exportar.")

✓ CSV guardado en: 7. Juni um 21-51_notas.csv


## 7. Reproducir las notas detectadas

Síntesis de las notas con armónicos (flautín) y reproductor integrado en el notebook.

In [ ]:
from IPython.display import Audio, display

SR = 22050  # Hz

def sintetizar_nota(freq, duracion_s, vel=0.5, sr=SR, fade=0.015):
    """Sine + 2 armónicos con envelope ADSR simple (imitación flauta)."""
    t = np.linspace(0, duracion_s, int(sr * duracion_s), endpoint=False)
    senal = (0.65 * np.sin(2 * np.pi * freq * t)
           + 0.25 * np.sin(2 * np.pi * 2 * freq * t)
           + 0.10 * np.sin(2 * np.pi * 3 * freq * t))
    senal *= float(vel)
    # fade in / fade out
    n_fade = int(fade * sr)
    if len(senal) > 2 * n_fade:
        senal[:n_fade]  *= np.linspace(0.0, 1.0, n_fade)
        senal[-n_fade:] *= np.linspace(1.0, 0.0, n_fade)
    return senal.astype(np.float32)

if note_events:
    total_dur = max(end for _, end, _, _, _ in note_events) + 0.5
    audio_out  = np.zeros(int(SR * total_dur), dtype=np.float32)

    for start, end, pitch, vel, _ in note_events:
        freq    = 440.0 * (2.0 ** ((pitch - 69) / 12.0))
        duracion = max(end - start, 0.05)
        senal   = sintetizar_nota(freq, duracion, vel=float(vel))
        i0 = int(start * SR)
        i1 = i0 + len(senal)
        if i1 <= len(audio_out):
            audio_out[i0:i1] += senal

    # Normalizar para evitar clipping
    mx = np.max(np.abs(audio_out))
    if mx > 0:
        audio_out /= mx

    print(f"▶ Reproduciendo {len(note_events)} notas ({total_dur:.1f} s) ...")
    display(Audio(audio_out, rate=SR, autoplay=False))
else:
    print("⚠ No hay notas para reproducir.")
